In [ ]:
PINS = __import__('json').loads("{\"baseModel\":\"openai/gpt-oss-20b\",\"baseModelRevision\":\"6cee5e81ee83917806bbde320786a8fb61efebee\",\"loaderModelId\":\"unsloth/gpt-oss-20b\",\"maxSeqLength\":1024,\"engineFreeze\":\"unsloth-freeze-2026.09.15\",\"engineDependencies\":[{\"name\":\"unsloth\",\"spec\":\"unsloth==2026.9.4\"},{\"name\":\"unsloth_zoo\",\"spec\":\"unsloth_zoo==2026.9.3\"},{\"name\":\"transformers\",\"spec\":\"transformers==4.56.2\"},{\"name\":\"peft\",\"spec\":\"peft==0.20.0\"},{\"name\":\"trl\",\"spec\":\"trl==0.22.2\"},{\"name\":\"datasets\",\"spec\":\"datasets==5.0.1\"},{\"name\":\"accelerate\",\"spec\":\"accelerate==1.15.0\"},{\"name\":\"bitsandbytes\",\"spec\":\"bitsandbytes==0.50.2\"},{\"name\":\"openai-harmony\",\"spec\":\"openai-harmony==0.0.8\"}],\"frozenNoDeps\":[\"unsloth\",\"unsloth_zoo\"],\"supportNoDeps\":[\"torchao>=0.16.0\"],\"preservedCandidates\":[\"torch\",\"triton\"],\"skipWhenPreserved\":[\"triton_kernels\"]}")
"""DEC-0037: one base-load diagnostic. No attached data or model execution."""
import json
import pathlib
import re
import traceback


def safe(text):
    return re.sub(r'hf_[A-Za-z0-9]+', '<redacted>', str(text))


def install():
    # Governed install source is injected unchanged by build-probe.mjs.
    # --- dependencies: the SAME governed engine stack the training run used ---
    #
    # Reproduces the accepted three-stage `uv` discipline (engine freeze
    # unsloth-freeze-2026.09.15). The ad-hoc %pip sequence this replaced was DEFECT 1 of the
    # pre-execution blocker recorded in DEC-0033 / BLK-0004: it submitted the whole
    # frozen set to ONE resolver transaction, which is unsatisfiable because
    # unsloth/unsloth_zoo cap `datasets<4.4.0` while the freeze pins `datasets==5.0.1`.
    #
    # Guarantees:
    #   1. No -qqq on install commands - complete stdout + stderr are captured.
    #   2. Every stage is dry-run with its EXACT arguments immediately before it runs.
    #   3. On failure a bounded redacted diagnostic is persisted, and the raised error
    #      carries the real resolver/package reason (never just 'exit code 1').
    #   4. Preinstalled torch/triton are preserved and constraint-pinned, so no stage
    #      can upgrade them off the +cu128 build Kaggle provides.
    #   5. triton_kernels is skipped on the Kaggle preserve path (upstream-aligned).
    import importlib.metadata as _metadata
    import os, pathlib, re, shutil, subprocess, sys
    
    WORKING = pathlib.Path('/kaggle/working')
    if not WORKING.exists():
        WORKING = pathlib.Path('.')
    
    INSTALL_DIAGNOSTIC_PATH = WORKING / 'eval-install-diagnostic.json'
    
    def redact(text):
        # Removes anything credential-shaped before it is printed or persisted.
        text = re.sub(r'hf_[A-Za-z0-9]{10,}', '<redacted-hf-token>', text)
        text = re.sub(r'(?i)(api[_-]?key|token|secret|password)([\"\']?\s*[:=]\s*)([^\s\"\',]+)',
                      r'\1\2<redacted>', text)
        return text
    
    def scrub_paths(text):
        return text.replace('/kaggle/input/', '/kaggle/input/<dataset>/')
    
    def run_install_command(cmd, phase, timeout=None):
        display = scrub_paths(redact(' '.join(cmd)))
        print('$', display)
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if proc.returncode != 0:
            max_diag = 32000
            out_trim = scrub_paths(redact(proc.stdout or ''))[-max_diag:]
            err_trim = scrub_paths(redact(proc.stderr or ''))[-max_diag:]
            diagnostic = {'phase': phase, 'command': display, 'exit_code': proc.returncode,
                          'stdout_redacted': out_trim, 'stderr_redacted': err_trim}
            try:
                INSTALL_DIAGNOSTIC_PATH.write_text(json.dumps(diagnostic, indent=1), encoding='utf-8')
            except Exception as exc:
                print('could not persist install diagnostic:', type(exc).__name__)
            raise RuntimeError(
                '%s failed (exit code %d).\n--- redacted stdout (last %d chars) ---\n%s\n'
                '--- redacted stderr (last %d chars) ---\n%s\nDiagnostic persisted to %s'
                % (phase, proc.returncode, len(out_trim), out_trim, len(err_trim), err_trim,
                   INSTALL_DIAGNOSTIC_PATH.name))
        return proc
    
    print('Bootstrapping uv...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '-qqq', 'uv'], check=True)
    UV = shutil.which('uv') or os.path.join(os.path.dirname(sys.executable), 'uv')
    if not shutil.which('uv') and not os.path.exists(UV):
        raise RuntimeError('uv was installed but is not on PATH - cannot continue.')
    
    if os.environ.get('VIRTUAL_ENV'):
        TARGET_FLAGS = ['--python', sys.executable]
    else:
        TARGET_FLAGS = ['--system', '--python', sys.executable]
    
    # Turing-only build target: keeps any source build from emitting sm_80+ kernels a
    # T4 cannot load.
    os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
    os.environ.setdefault('CMAKE_CUDA_ARCHITECTURES', '75')
    
    ON_KAGGLE = os.path.isdir('/kaggle')
    
    def module_present(modname):
        try:
            __import__(modname)
            return True
        except Exception:
            return False
    
    PRESERVED = {}
    if ON_KAGGLE:
        for _name in PINS['preservedCandidates']:
            if module_present(_name):
                PRESERVED[_name] = _metadata.version(_name)
    
    SKIPPED = set()
    if PRESERVED:
        SKIPPED.update(PINS['skipWhenPreserved'])
    
    resolver_specs = []
    frozen_specs = []
    for _dep in PINS['engineDependencies']:
        if _dep['name'] in PRESERVED:
            print('preserving preinstalled %s==%s (not re-resolved against PyPI)'
                  % (_dep['name'], PRESERVED[_dep['name']]))
            continue
        if _dep['name'] in SKIPPED:
            print('skipping %s on the Kaggle preserve path (upstream-aligned)' % _dep['name'])
            continue
        if _dep['name'] in PINS['frozenNoDeps']:
            frozen_specs.append(_dep['spec'])
        else:
            resolver_specs.append(_dep['spec'])
    
    print('Stage 1 (resolver-managed):', resolver_specs)
    print('Stage 2 (--no-deps frozen set):', frozen_specs)
    print('Stage 3 (--no-deps support):', PINS['supportNoDeps'])
    
    # Exact constraints stop any transitive dependency from upgrading the preserved builds.
    CONSTRAINT_PATH = WORKING / 'eval-preserved-constraints.txt'
    CONSTRAINT_PATH.write_text(''.join('%s==%s\n' % _i for _i in sorted(PRESERVED.items())),
                               encoding='utf-8')
    CONSTRAINT_FLAGS = ['--constraint', str(CONSTRAINT_PATH)] if PRESERVED else []
    
    BASE = [UV, 'pip', 'install', *TARGET_FLAGS, '--no-cache-dir', *CONSTRAINT_FLAGS]
    
    INSTALL_PLAN = [
        ('install', [*BASE, *resolver_specs]),
        ('frozen-no-deps', [*BASE, '--upgrade', '--no-deps', *frozen_specs]),
        ('support-no-deps', [*BASE, '--no-deps', '--upgrade', *PINS['supportNoDeps']]),
    ]
    
    for _phase, _cmd in INSTALL_PLAN:
        run_install_command([*_cmd, '--dry-run'], _phase + '-dry-run')
        run_install_command(_cmd, _phase)
    
    for _name, _version in PRESERVED.items():
        _actual = _metadata.version(_name)
        if _actual != _version:
            raise RuntimeError('preserved dependency changed: %s==%s -> %s'
                               % (_name, _version, _actual))
    
    print('install complete')
    print('preserved:', PRESERVED or 'none')
    print('skipped:', sorted(SKIPPED) or 'none')


def main():
    phase = 'INPUT_ISOLATION'
    result = {'test_accessed': False, 'inference_executed': False,
              'install': False, 'tokenizer': False, 'model_load': False,
              'base_revision': None, 'status': 'BLOCKED'}
    try:
        # Inspect directory entries only; never open an input file.
        inputs = pathlib.Path('/kaggle/input')
        assert not inputs.exists() or not any(inputs.iterdir()), 'Attached input forbidden'
        phase = 'INSTALL'
        install()
        import importlib.metadata as metadata
        for dependency in PINS['engineDependencies']:
            expected = dependency['spec'].split('==', 1)[1]
            assert metadata.version(dependency['name']) == expected, dependency['name']
        result['install'] = True
        print('DIAGNOSTIC_INSTALL_PASS', flush=True)

        phase = 'BASE_REVISION'
        from huggingface_hub import HfApi
        api = HfApi()
        revision = api.model_info(PINS['baseModel']).sha
        assert revision == PINS['baseModelRevision'], 'Upstream base revision drift'
        result['base_revision'] = revision
        print('DIAGNOSTIC_BASE_REVISION_VERIFIED', revision, flush=True)
        # This verifies the governed upstream reference, not a weight equivalence proof
        # for the separately quantized Unsloth distribution. Print both identities.
        distribution = PINS['loaderModelId'] + '-unsloth-bnb-4bit'
        dist_info = api.model_info(distribution)
        result['distribution_revision'] = dist_info.sha
        print('DISTRIBUTION_IDENTITY', distribution, dist_info.sha, flush=True)
        # Separate metadata requests distinguish missing optional folder from download
        # errors. These are diagnostic observations, never load-path patches.
        for path in ('additional_chat_templates', ''):
            try:
                entries = list(api.list_repo_tree(distribution, path_in_repo=path,
                                                  revision=dist_info.sha))
                print('HUB_TREE', path or '<root>', 'OK', len(entries), flush=True)
            except Exception as exc:
                print('HUB_TREE', path or '<root>', type(exc).__name__, safe(exc), flush=True)

        phase = 'MODEL_TOKENIZER_LOAD'
        from unsloth import FastLanguageModel
        # Observe tokenizer/processor failures before Unsloth wraps the exception.
        from transformers import AutoTokenizer, AutoProcessor
        def observe(original, label):
            def load(*args, **kwargs):
                try:
                    obj = original(*args, **kwargs)
                    print('LOADER_COMPONENT_LOADED', label, type(obj).__name__, flush=True)
                    return obj
                except Exception:
                    print('LOADER_COMPONENT_TRACEBACK', label, safe(traceback.format_exc()), flush=True)
                    raise
            return load
        AutoTokenizer.from_pretrained = observe(AutoTokenizer.from_pretrained, 'AutoTokenizer')
        AutoProcessor.from_pretrained = observe(AutoProcessor.from_pretrained, 'AutoProcessor')
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=PINS['loaderModelId'], max_seq_length=PINS['maxSeqLength'],
            dtype=None, load_in_4bit=True,
        )
        assert model is not None and tokenizer is not None
        result['model_load'] = True
        result['tokenizer'] = True
        result['model_class'] = type(model).__name__
        result['tokenizer_class'] = type(tokenizer).__name__
        result['loaded_model_id'] = str(model.config._name_or_path)
        result['loaded_model_revision'] = getattr(model.config, '_commit_hash', None)
        result['tokenizer_id'] = str(getattr(tokenizer, 'name_or_path', ''))
        print('DIAGNOSTIC_TOKENIZER_PASS', flush=True)
        print('DIAGNOSTIC_MODEL_LOAD_PASS', flush=True)
        phase = 'LOADED_IDENTITY'
        assert result['loaded_model_id'] in (PINS['loaderModelId'], distribution)
        assert result['loaded_model_revision'] == dist_info.sha, 'Distribution revision changed during load'
        assert result['tokenizer_id'] in (PINS['loaderModelId'], distribution)
        print('VERIFIED_IDENTITY', json.dumps(result, sort_keys=True), flush=True)
        result['status'] = 'PASS'
    except Exception as exc:
        result['failure_phase'] = phase
        result['exception'] = type(exc).__name__
        result['traceback'] = safe(traceback.format_exc())
        result['failure_class'] = ('TOKENIZER_PROCESSOR_LOAD' if 'tokenizer/processor' in str(exc)
                                   else phase)
        print('DIAGNOSTIC_TRACEBACK', result['traceback'], flush=True)
        raise
    finally:
        print('NO_TEST_ACCESSED', flush=True)
        print('NO_INFERENCE_EXECUTED', flush=True)
        pathlib.Path('/kaggle/working/diagnostic-result.json').write_text(
            json.dumps(result, indent=2), encoding='utf-8')
        if result['status'] == 'PASS':
            print('DIAGNOSTIC_COMPLETE', flush=True)


if __name__ == '__main__':
    main()
